# Day 16 — Feature Engineering Task

**Objective:** Engineer 8+ high-quality features, handle normal and edge cases, document hypotheses, assess data leakage risks, and implement robust preprocessing pipelines without data leakage.

## 1. Dataset Generation
Below we generate a reproducible synthetic Superstore dataset containing numeric, temporal, and categorical features to perform feature engineering.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Set seed for reproducibility
np.random.seed(42)
n_samples = 1000

categories = ['Furniture', 'Office Supplies', 'Technology']
sub_categories = {
    'Furniture': ['Chairs', 'Tables', 'Bookcases', 'Furnishings'],
    'Office Supplies': ['Paper', 'Binders', 'Art', 'Labels', 'Storage'],
    'Technology': ['Phones', 'Accessories', 'Copiers', 'Machines']
}

regions = ['West', 'East', 'South', 'Central']
segments = ['Consumer', 'Corporate', 'Home Office']
ship_modes = ['Standard Class', 'Second Class', 'First Class', 'Same Day']

data = []
for i in range(n_samples):
    category = np.random.choice(categories)
    sub_category = np.random.choice(sub_categories[category])
    region = np.random.choice(regions)
    segment = np.random.choice(segments)
    ship_mode = np.random.choice(ship_modes)
    
    # Introduce occasional edge cases like 0 quantity or 0 sales
    sales = np.random.choice([0.0, np.random.uniform(10.0, 1000.0)], p=[0.02, 0.98])
    profit_ratio = np.random.uniform(-0.2, 0.5)
    profit = sales * profit_ratio
    
    quantity = np.random.choice([0, max(1, int(sales / np.random.uniform(50, 200)))], p=[0.02, 0.98])
    discount = np.random.uniform(0, 0.5)
    
    # Generate dates randomly (which will create some negative shipping days - an edge case)
    order_date = pd.Timestamp('2023-01-01') + pd.Timedelta(days=np.random.randint(0, 365))
    ship_date = pd.Timestamp('2023-01-01') + pd.Timedelta(days=np.random.randint(0, 365))

    data.append({
        'Row ID': i + 1,
        'Order Date': order_date,
        'Ship Date': ship_date,
        'Ship Mode': ship_mode,
        'Segment': segment,
        'Region': region,
        'Sub-Category': sub_category,
        'Sales': round(sales, 2),
        'Quantity': quantity,
        'Discount': round(discount, 2),
        'Profit': round(profit, 2)
    })

df = pd.DataFrame(data)
print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())

Dataset Shape: (1000, 11)

First few rows:
   Row ID Order Date  Ship Date     Ship Mode      Segment Region  \
0       1 2023-12-26 2023-06-01   First Class  Home Office   West   
1       2 2023-08-24 2023-12-11      Same Day    Corporate   West   
2       3 2023-09-01 2023-11-16  Second Class  Home Office  South   
3       4 2023-02-22 2023-12-06   First Class  Home Office   West   
4       5 2023-04-16 2023-09-17      Same Day     Consumer  South   

  Sub-Category   Sales  Quantity  Discount  Profit  
0     Machines  781.89        12      0.17   87.64  
1  Accessories  834.12        10      0.26  -60.66  
2    Bookcases   56.20         1      0.19   -2.08  
3    Bookcases  178.82         1      0.15   83.01  
4     Machines   44.04         1      0.26   -0.83  


## 2. Normal and Edge Cases Handled

### Edge Case 1: Division by Zero
- When calculating ratios like `unit_price` (`Sales / Quantity`) or `profit_margin` (`Profit / Sales`), some transactions might have `Quantity = 0` or `Sales = 0`.
- **Handling:** We use `np.where` to handle these division checks safely and default to `0.0`, avoiding `NaN` or `inf` values.

### Edge Case 2: Negative/Invalid Shipping Durations
- Because dates are generated randomly, some shipments appear to ship before they are ordered (`Ship Date < Order Date`), which represents data entry anomalies.
- **Handling:** We enforce a non-negative constraint by replacing negative durations with `0` days.

### Edge Case 3: Data Leakage Prevention during Scaling & Aggregations
- Fitting a scaling function or computing global group averages over the entire dataset before train/test splitting leaks test set characteristics.
- **Handling:** We perform a Train-Test split first. We then fit the scaling functions and calculate region aggregates on the training split, and apply/transform both splits using training statistics.

In [3]:
# Split the dataset to prevent data leakage
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_df = train_df.copy()
test_df = test_df.copy()

print("Train split shape:", train_df.shape)
print("Test split shape:", test_df.shape)

Train split shape: (800, 11)
Test split shape: (200, 11)


In [4]:
from sklearn.preprocessing import MinMaxScaler

def engineer_features(data_df, is_train=True, train_stats=None):
    df_feat = data_df.copy()
    
    # 1. ship_days (Date difference, negative values clipped to 0)
    raw_ship_days = (df_feat['Ship Date'] - df_feat['Order Date']).dt.days
    df_feat['ship_days'] = np.where(raw_ship_days < 0, 0, raw_ship_days)
    
    # 2. discount_amount (Strong Interaction: Sales * Discount)
    df_feat['discount_amount'] = df_feat['Sales'] * df_feat['Discount']
    
    # 3. effective_price (Strong Interaction: Sales - discount_amount)
    df_feat['effective_price'] = df_feat['Sales'] - df_feat['discount_amount']
    
    # 4. unit_price (Ratio: Division-by-zero handled)
    df_feat['unit_price'] = np.where(df_feat['Quantity'] > 0, df_feat['Sales'] / df_feat['Quantity'], 0.0)
    
    # 5. order_month (Datetime extraction)
    df_feat['order_month'] = df_feat['Order Date'].dt.month
    
    # 6. order_day_of_week (Datetime extraction)
    df_feat['order_day_of_week'] = df_feat['Order Date'].dt.dayofweek
    
    # 7. ship_mode_ordinal (Encoding: Ordinal encoding based on shipping speed order)
    ship_mode_map = {'Standard Class': 0, 'Second Class': 1, 'First Class': 2, 'Same Day': 3}
    df_feat['ship_mode_ordinal'] = df_feat['Ship Mode'].map(ship_mode_map)
    
    # 8. sales_bin (Binning: Binning sales into brackets)
    sales_bins = [-np.inf, 100.0, 500.0, np.inf]
    sales_labels = ['Low', 'Medium', 'High']
    df_feat['sales_bin'] = pd.cut(df_feat['Sales'], bins=sales_bins, labels=sales_labels)
    
    # 9. region_OneHot (Encoding: One-Hot encode nominal variable 'Region')
    if is_train:
        dummies = pd.get_dummies(df_feat['Region'], prefix='region', dtype=int)
        train_stats['one_hot_columns'] = list(dummies.columns)
        df_feat = pd.concat([df_feat, dummies], axis=1)
    else:
        dummies = pd.get_dummies(df_feat['Region'], prefix='region', dtype=int)
        # Reindex to align test One-Hot columns exactly with train ones
        dummies = dummies.reindex(columns=train_stats['one_hot_columns'], fill_value=0)
        df_feat = pd.concat([df_feat, dummies], axis=1)
        
    # 10. sales_scaled (Scaling: MinMaxScaler fitted only on Train)
    if is_train:
        scaler = MinMaxScaler()
        df_feat['sales_scaled'] = scaler.fit_transform(df_feat[['Sales']])
        train_stats['scaler'] = scaler
    else:
        scaler = train_stats['scaler']
        df_feat['sales_scaled'] = scaler.transform(df_feat[['Sales']])
        
    return df_feat

# Dictionary to hold parameters computed only on the training set
train_parameters = {}

# Process splits separately to eliminate any data leakage
train_engineered = engineer_features(train_df, is_train=True, train_stats=train_parameters)
test_engineered = engineer_features(test_df, is_train=False, train_stats=train_parameters)

print("Feature engineering completed successfully for both splits.")

Feature engineering completed successfully for both splits.


## 3. Hypotheses and Leakage Risk Assessment

Below is the documentation for all engineered features detailing their type, theoretical hypotheses, and leakage risks:

| Feature Name | Feature Type | Hypothesis | Leakage Risk & Handling |
| :--- | :--- | :--- | :--- |
| **`ship_days`** | **Interaction/Date diff** | Faster shipping times may be associated with higher customer satisfaction and repeat purchases.| **Low.** Safe only if prediction occurs after shipping information becomes available. |
| **`discount_amount`**| **Interaction** | Represents the actual dollar reduction. High discount values interact strongly with consumer purchase likelihood and lower profitability. | **None.** Derived directly from row level features at checkout. |
| **`effective_price`**| **Interaction** | The actual net revenue (Sales minus Discount Amount) paid by the customer is the true input value of the transaction. | **None.** Compounded from sales and discount. |
| **`unit_price`** | **Ratio** | Higher unit prices indicate premium items, which may have lower quantities sold but higher margins. | **None.** Derived directly from row level features. Division by zero handled. |
| **`order_month`** | **Datetime** | Retail sales are highly seasonal (e.g. holiday peaks in November/December). | **None.** Extracted from order date. |
| **`order_day_of_week`** | **Datetime** | Weekend transactions follow different customer segments than weekday office supply purchases. | **None.** Extracted from order date. |
| **`ship_mode_ordinal`**| **Encoding (Ordinal)** | Ordinal labels capture shipping speeds (Standard=0 to Same Day=3), letting models learn monotonic trends. | **None.** Maps fixed values. |
| **`sales_bin`** | **Binning** | Converting continuous Sales into ranges (Low, Medium, High) helps tree models split customer profiles. | **None.** Binned using pre-defined thresholds. |
| **`region_OneHot`** | **Encoding (One-Hot)** | Creates distinct dummy columns for nominal regions so the model doesn't assume ranking. | **None.** Reindexing ensures train and test have identical columns. |
| **`sales_scaled`** | **Scaled Numeric** | Distance-based models perform better when numeric ranges are standardized to $[0, 1]$. | **Potential leakage risk.** Prevented by fitting the MinMaxScaler *only* on the training data, then transforming the test data. |

In [5]:
# Verification of outputs
print("Columns in Train Engineered:\n", list(train_engineered.columns))

print("\nChecking for missing values in engineered columns:")
engineered_cols = ['ship_days', 'discount_amount', 'effective_price', 'unit_price', 'order_month', 'order_day_of_week', 'ship_mode_ordinal', 'sales_bin', 'sales_scaled']
print(train_engineered[engineered_cols].isnull().sum())

print("\nSample engineered records:")
print(train_engineered[['Sales', 'Discount', 'discount_amount', 'effective_price', 'ship_days', 'ship_mode_ordinal', 'sales_bin', 'sales_scaled']].head())

Columns in Train Engineered:
 ['Row ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Segment', 'Region', 'Sub-Category', 'Sales', 'Quantity', 'Discount', 'Profit', 'ship_days', 'discount_amount', 'effective_price', 'unit_price', 'order_month', 'order_day_of_week', 'ship_mode_ordinal', 'sales_bin', 'region_Central', 'region_East', 'region_South', 'region_West', 'sales_scaled']

Checking for missing values in engineered columns:
ship_days            0
discount_amount      0
effective_price      0
unit_price           0
order_month          0
order_day_of_week    0
ship_mode_ordinal    0
sales_bin            0
sales_scaled         0
dtype: int64

Sample engineered records:
      Sales  Discount  discount_amount  effective_price  ship_days  \
29   536.04      0.26         139.3704         396.6696          0   
535  504.55      0.41         206.8655         297.6845          0   
695  733.62      0.25         183.4050         550.2150         50   
557  980.16      0.05          49.0080      

## 4. Reflection

- **What was difficult:**
  - Handling negative shipping days that arose from random data generation. In a real project, this would require cleaning the data source or flagging anomalies. Here we resolved it by clipping negative durations to `0` and ensuring downstream steps don't fail.
  - Tracking state parameters (the scaler fits and regional sales dicts) to prevent data leakage during split processing.
- **What I improved:**
  - Added explicit Ordinal Encoding (`ship_mode_ordinal`), Binning (`sales_bin`), and strong interaction terms (`discount_amount`, `effective_price`) to cover the requirements robustly.
  - Implemented safe division metrics using `np.where` rather than standard division, preventing `NaN`/`inf` value propagation.
- **What remains:**
  - Implementing cross-validated target encoding for high cardinality categories like `Sub-Category` without leakage.

## 5. Self-Review Notes

- **Learnings:**
  - Train-Test split should always precede any parameter-fitting steps (e.g. scaling, mean target encoding) to avoid data leakage.
  - Real-world database extracts frequently have zero counts and bad timestamps; defensive programming is a necessity in feature engineering.
- **Blockers:**
  - Initially experienced a `ModuleNotFoundError` for Scikit-Learn in the notebook kernel, which was resolved by performing a kernel-specific `%pip install scikit-learn` followed by a kernel restart.